# ROS 2 Nav2 — Localization & Mapping

---

## Visualizing the Map

In this directory, the file `map.pgm` contains the occupancy grid map of the simulated environment. Alongside it, the `map.yaml` file stores important information about the map, such as its resolution, origin, and the location of the corresponding `.pgm` file.

ROS 2 uses both files together to correctly load and interpret the map during localization and navigation.

### Map of the Simulated Environment

![Map of the simulated environment](./map.png)

## Launching Localization with Nav2

First, launch Gazebo with the MIRTE robot in one terminal:

```bash
ros2 launch spatial_ai_simulation spatial_ai_mirte_master.launch.py
```

You should see the simulated world with the MIRTE robot loaded in Gazebo.

![MIRTE robot in Gazebo](./mirte_gazebo.png)

Next, open RViz in a second terminal:

```bash
rviz2
```

In a third terminal, launch Nav2 localization using the map file:

```bash
ros2 launch nav2_bringup localization_launch.py map:=./map.yaml use_sim_time:=true
```

You may see an error similar to this:

```
[amcl-2] [ERROR] [1780240116.450536610] [amcl]: Couldn't transform from lidar_link to base_footprint,
even though the message notifier is in use: ("base_footprint" passed to lookupTransform argument
target_frame does not exist.)
```

This happens because Nav2 expects the robot base frame to be named `base_footprint`, while the MIRTE robot uses `base_link`.

Instead of manually changing the frame name in multiple configuration files, we can publish a static transform between `base_link` and `base_footprint`. Since both frames represent the same robot base position in this setup, the transform can be set to zero translation and zero rotation:

```bash
ros2 run tf2_ros static_transform_publisher 0 0 0 0 0 0 base_link base_footprint
```

After running this command, Nav2 should be able to transform between the required frames and continue localization.

In the RViz environment, add the topic `map` and make sure the fixed frame is set to `map`.

In case you have a warning in the status that says **no map received**, go to the topic section and change the durability policy from **Volatile** to **Transient Local**. You should now be able to see the map loaded in RViz.

## Initial Pose Estimate

First, let's see our `/amcl_pose` topic by adding it in RViz. If you add it you will see that nothing is shown. That is because if you go to the terminal where you launched the localization node you should see something like this:

```
[amcl-2] [WARN] [1780240686.938755585] [amcl]: AMCL cannot publish a pose or update the transform.
Please set the initial pose...
```

So let's do exactly that. Go to RViz, click **2D Pose Estimate**, and with Gazebo open place an initial position of the robot on the RViz map. Make sure to drag the arrow to also define the orientation of the robot.

![Initial pose estimate](./initial_pose_estimate.png)

In [ ]:
import hashlib

def ask(question, options, answer_hash, feedback):
    print(question)
    print()
    for letter, option in zip(['A', 'B', 'C', 'D'], options):
        print(f"  {letter}) {option}")
    print()
    answer = input("Your answer (A/B/C/D): ").strip().upper()
    print()
    if hashlib.md5(answer.encode()).hexdigest() == answer_hash:
        print("✓ Correct!")
        print(f"→ {feedback}")
    else:
        print("✗ Not quite — try again or move on and revisit later.")

print("Helper loaded.")

In [ ]:
ask(
    "Q1. In RViz, after setting an initial pose estimate, what does the purple circle of particles represent?",
    [
        "The global costmap boundary",
        "The set of candidate robot poses — each particle is a hypothesis about where the robot could be",
        "The inflation radius around obstacles",
        "The laser scan range ring"
    ],
    "9d5ed678fe57bcca610140957afab571",
    "The purple particle cloud is AMCL's belief about the robot's possible poses. Each particle is one "
    "hypothesis; collectively they form a probability distribution over the robot's location."
)

In [ ]:
ask(
    "Q2. What does the yellow cone in the RViz initial pose visualization represent?",
    [
        "The robot's goal heading",
        "The current velocity vector of the robot",
        "The estimated orientation of the robot (the direction the robot is facing)",
        "A warning that no map has been received"
    ],
    "0d61f8370cad1d412f80b84d143e1257",
    "The yellow cone shows the estimated orientation provided when you dragged the 2D Pose Estimate arrow. "
    "It represents the robot's heading direction."
)

## Navigating with Teleop

Let's navigate around with the robot using the teleop we used in the workshop where we got introduced to the robot.

Launch teleop like this:

```bash
ros2 launch mirte_teleop teleop_key.launch.py
```

Navigate around the map and see what happens as you drive toward the purple circle and the yellow cone.

In [ ]:
ask(
    "Q3. As you drive the robot around with teleop and observe the particle cloud, what happens over time?",
    [
        "The particle cloud grows larger as the robot travels further from the start",
        "The particle cloud stays the same size regardless of movement",
        "The particle cloud converges and shrinks as the robot gathers more sensor data that disambiguates its position",
        "The particle cloud disappears once the robot leaves its starting area"
    ],
    "0d61f8370cad1d412f80b84d143e1257",
    "As the robot moves and accumulates laser scan evidence that matches the map, AMCL reweights and "
    "resamples particles — low-probability particles are discarded and the cloud converges toward the true pose."
)

In [ ]:
ask(
    "Q4. Why is the particle cloud larger in some parts of the map and smaller in others?",
    [
        "It reflects the Wi-Fi signal strength in that area",
        "It depends on how fast the robot is moving through that area",
        "Symmetric or featureless areas cause ambiguity in sensor readings, making it harder to "
        "distinguish one location from another, so uncertainty stays high",
        "It is proportional to the distance from the map origin"
    ],
    "0d61f8370cad1d412f80b84d143e1257",
    "In geometrically symmetric corridors or open spaces, the laser scans look similar from multiple "
    "positions, so AMCL can't rule out many hypotheses and the cloud remains spread out. Distinctive "
    "features (corners, alcoves) constrain the estimate and shrink the cloud."
)

## AMCL Parameters

AMCL is a probabilistic localization system for a robot moving in 2D. It implements the adaptive (or KLD-sampling) Monte Carlo localization approach, which uses a particle filter to track the pose of a robot against a known map.

Localization was explained in one of your lectures during the flipped classroom, so the questions below can be answered with a simple understanding. Feel free to also go through this explanation of how it works:

https://roboticsknowledgebase.com/wiki/state-estimation/adaptive-monte-carlo-localization/

In [ ]:
ask(
    "Q5. Your robot is not localizing correctly despite a good initial pose estimate. "
    "Which AMCL parameter adjustment is most likely to improve performance?",
    [
        "Increase the map resolution in map.yaml",
        "Increase max_particles and tune the motion model noise parameters (alpha1-alpha4) "
        "to better match the real odometry noise",
        "Switch from use_sim_time:=true to use_sim_time:=false",
        "Reduce the scan topic frequency to save CPU"
    ],
    "9d5ed678fe57bcca610140957afab571",
    "AMCL's accuracy depends heavily on having enough particles to represent the distribution and on motion "
    "model noise parameters that realistically reflect odometry drift. Tuning alpha1-alpha4 and increasing "
    "max_particles are the primary levers for localization quality."
)

In [ ]:
ask(
    "Q6. After a long run, the robot appears in two locations on the map simultaneously. "
    "What is the most likely cause and fix?",
    [
        "RViz is rendering two TF frames at once; fix by restarting RViz",
        "The map file has a duplicate entry; regenerate the map",
        "AMCL has a multimodal particle distribution — the filter is tracking two competing hypotheses, "
        "often due to symmetry. Increase min_particles, tune recovery parameters, or manually re-initialize the pose",
        "The robot's odometry topic is being published on two different namespaces"
    ],
    "0d61f8370cad1d412f80b84d143e1257",
    "When the environment is symmetric, AMCL can split its particle weight across two equally plausible poses. "
    "The fix is to navigate the robot to a distinctive area, re-initialize the pose estimate, or tune "
    "recovery_alpha parameters to inject random particles and escape the local mode."
)

## Wrong Initial Pose Estimate

Now let's give the robot a **wrong** initial pose estimate and navigate around using teleop. Observe what happens — notice how the particle cloud behaves as you drive around.

In [ ]:
ask(
    "Q7. You gave the robot a wrong initial pose estimate and start driving with teleop. "
    "What happens as the robot approaches the area matching its actual position in the map?",
    [
        "The particle cloud jumps instantly to the correct pose as soon as any movement occurs",
        "Nothing changes — AMCL is locked to the initial estimate until you manually re-initialize",
        "The particle cloud gradually shifts and converges toward the correct location as sensor data "
        "from the real environment increasingly matches that region of the map",
        "The robot stops moving because the costmap detects an invalid pose"
    ],
    "0d61f8370cad1d412f80b84d143e1257",
    "AMCL continually reweights particles based on laser scan likelihood. As the robot moves into a region "
    "whose sensor signature matches the particles near the true pose, those particles accumulate weight and "
    "the cloud migrates and converges to the correct location — even recovering from a bad initial estimate."
)

## Local Costmap

To see local costmaps, launch the navigation stack in a different terminal:

```bash
ros2 launch nav2_bringup navigation_launch.py \
  map:=./map.yaml \
  params_file:=./nav2_params.yaml \
  use_sim_time:=true
```

Then add the relevant topics in RViz. You should be able to see the local costmap overlaid on the map.

![Local costmap](./local_costmap.png)

## Mapping with SLAM

Now that was cool — but as you already know, SLAM stands for **Simultaneous Localization and Mapping**, so let's do exactly that in order to obtain our own map (like the one used at the beginning).

```bash
ros2 launch slam_toolbox online_async_launch.py use_sim_time:=true
```

Reopen RViz and show the `map` topic again. Now navigate around the map and watch it being built in real time. It is really important for navigation and localization for your map to be as accurate as possible, so make sure to navigate to every position that holds useful information.

When you are done, save the map:

```bash
ros2 run nav2_map_server map_saver_cli -f my_map
```

When running the localization and navigation commands above, make sure to replace `./map.yaml` with your own saved map file.

---

**That's it for localization and mapping!**